# 01 FTH — image series / hysteresis loop

Apply the calibration from the original `01_FTH.ipynb` (and, when available,
`03_define_supportmask.ipynb`) to many `+`/`-` image pairs. The calibrated
image-13 center, pixel mask, smooth beamstop mask, Ewald setting, FTH propagation,
global phase, and ROI are reused unchanged.

Run the original notebooks on one representative pair first. Edit only the
configuration cell below for a new series.

In [ ]:
# Configure Qt before importing pyplot or any other GUI-aware library.
import os
import sys
from os.path import join

BASEFOLDER = os.path.abspath(os.getcwd())
LIBRARY_FOLDER = join(BASEFOLDER, "library")
if LIBRARY_FOLDER not in sys.path:
    sys.path.insert(0, LIBRARY_FOLDER)

from notebook_setup import configure_matplotlib_qt
MATPLOTLIB_BACKEND = configure_matplotlib_qt()
print("Matplotlib backend:", MATPLOTLIB_BACKEND)

import os, sys
from os.path import join

import numpy as np
import matplotlib.pyplot as plt

BASEFOLDER = os.path.abspath(os.getcwd())
sys.path.insert(0, join(BASEFOLDER, "library"))

import importlib
import CCI_core as cci
from interactive import cimshow
import fthcore as fth
import fth_phase_workflow as wf
wf = importlib.reload(wf)  # Refresh helpers in an existing kernel.
from data_loading import SextantsNexusLoader, image_ids, load_average

try:
    import Phase_Retrieval as PhR
except Exception:
    PhR = None

print("Base folder:", BASEFOLDER)
from copy import deepcopy
from pipeline_io import acquisition_ids as image_ids, first_id, result_path

from image_preprocessing import fit_dark_frame, fit_horizontal_band


## Configuration

In [ ]:
USER = "rb"
RAW_FOLDER = "/nfs/ruche/sextants-soleil/com-sextants/COMET_20261110_Battistelli_laser_RF/"

# First positive ID of a complete 01_FTH calibration. It remains untouched.
REFERENCE_IMAGE_ID = 1564
REFERENCE_IMAGE_ID = first_id(REFERENCE_IMAGE_ID)
REFERENCE_H5 = result_path(BASEFOLDER, REFERENCE_IMAGE_ID, USER)

# Scalars describe one series point; a flat list describes separate points.
# For an averaged point use a nested list, e.g. [[1564, 1566], [1568, 1570]].
PLUS_IMAGE_IDS = list(np.arange(1564, 1594, 2))
MINUS_IMAGE_IDS = list(np.arange(1565, 1595, 2))
SERIES_POINTS = [70]*3 + [75]*3 + [80]*3 + [90]*3 + [100]*3

if np.isscalar(PLUS_IMAGE_IDS):
    PLUS_IMAGE_IDS = [PLUS_IMAGE_IDS]
if np.isscalar(MINUS_IMAGE_IDS):
    MINUS_IMAGE_IDS = [MINUS_IMAGE_IDS]
if len(PLUS_IMAGE_IDS) != len(MINUS_IMAGE_IDS) or not len(PLUS_IMAGE_IDS):
    raise ValueError("Define equal, non-empty positive/negative series lists")
if SERIES_POINTS is None:
    SERIES_POINTS = list(range(len(PLUS_IMAGE_IDS)))
if len(SERIES_POINTS) != len(PLUS_IMAGE_IDS):
    raise ValueError("SERIES_POINTS must have one value per pair")
SERIES = [{"point": point, "+": image_ids(plus), "-": image_ids(minus)}
          for point, plus, minus in zip(SERIES_POINTS, PLUS_IMAGE_IDS, MINUS_IMAGE_IDS)]
DARK_IDS = None  # Inherit the calibration dark IDs; set a dict to override.
REUSE_CALIBRATION_SCALING = True
CENTER_OVERRIDE = None
ROI_OVERRIDE = None
GIF_FRAME_DURATION_MS = 250
OVERWRITE = True

SERIES_IM_ID = first_id(SERIES[0]["+"])
SERIES_H5 = join(BASEFOLDER, "processed", "Logs",
                 f"data_recon_series_ImId_{SERIES_IM_ID:04d}_{USER}_all.hdf5")
keys = [first_id(row["+"]) for row in SERIES]
if len(keys) != len(set(keys)):
    raise ValueError("Series entries must have distinct first positive IDs to avoid overwriting")


## Load and validate the calibration

In [451]:
calibration = wf.load_data_dict(REFERENCE_H5)
labels = ["+", "-"]
if any(label not in calibration["holo"] for label in labels):
    raise ValueError(f"Calibration must contain + and - modes; found {list(calibration['holo'])}")

saved_reference_id = int(image_ids(calibration["holo"]["+"]["id"])[0])
if saved_reference_id != REFERENCE_IMAGE_ID:
    raise ValueError(
        f"Reference file contains positive image {saved_reference_id}, "
        f"not requested image {REFERENCE_IMAGE_ID}"
    )
stored_center = np.asarray(calibration["center"], dtype=float)
center = stored_center if CENTER_OVERRIDE is None else np.asarray(CENTER_OVERRIDE, dtype=float)
experimental_setup = dict(calibration["experimental_setup"])
mask_pixel = np.asarray(calibration["mask_pixel"], dtype=np.uint8)
beamstop_recipe = dict(
    calibration.get("mask_beamstop_smooth_recipe")
    or calibration.get("mask_pixel_smooth_recipe", {"radius": 35, "order": 4})
)
mask_beamstop_smooth = wf.butterworth_disk_mask(
    mask_pixel.shape, beamstop_recipe["radius"], beamstop_recipe["order"]
)
mask_pixel_fth_recipe = dict(
    calibration.get("mask_pixel_fth_recipe", {"dilation_pixels": 3, "sigma": 3})
)
mask_pixel_fth = wf.smooth_binary_mask(
    mask_pixel.astype(float),
    mask_pixel_fth_recipe["dilation_pixels"],
    mask_pixel_fth_recipe["sigma"],
)
mask_multiplier = (1 - mask_beamstop_smooth) * (1 - mask_pixel_fth)
focus_fth = dict(calibration.get("focus_fth", {}))
prop_dist = float(focus_fth.get("prop_dist", 0))  # micrometres
prop_dist_unit = focus_fth.get("prop_dist_unit", "um")
if prop_dist_unit not in ("um", "µm"):
    raise ValueError(f"Expected FTH propagation in micrometres, got {prop_dist_unit!r}")
phase = float(focus_fth.get("phase", 0))
dx = float(focus_fth.get("dx", 0))
dy = float(focus_fth.get("dy", 0))
stored_roi = focus_fth.get("roi")
if stored_roi is None and ROI_OVERRIDE is None:
    raise ValueError("The reference has no saved ROI; finish 01_FTH.ipynb first")
roi = list(stored_roi if ROI_OVERRIDE is None else ROI_OVERRIDE)
roi_s = np.s_[roi[0] : roi[1], roi[2] : roi[3]]
focus_fth["roi"] = roi
project_ewalds_sphere = bool(calibration.get("project_ewalds_sphere", False))
ewald_method = calibration.get("ewald_method", "cubic")

if project_ewalds_sphere and PhR is None:
    raise RuntimeError("Calibration requests Ewald projection, but Phase_Retrieval could not import")
if tuple(mask_pixel.shape) != tuple(calibration["holo"]["+"]["image_c"].shape):
    raise ValueError("Calibration mask and centered hologram shapes differ")

print("Reference image:", REFERENCE_IMAGE_ID)
print("Reference HDF5:", REFERENCE_H5)
print("Center:", center, "mask pixels:", int(mask_pixel.sum()), "ROI:", roi)
print("FTH focus: prop_dist=", prop_dist, "phase=", phase)

KeyError: 'center'

## Process every image pair

In [447]:
fig,ax=plt.subplots()

import mask_lib
mask11=1-mask_lib.circle_mask(mask_pixel.shape,
                              center=[mask_pixel.shape[0]//2,mask_pixel.shape[0]//2],
                            radius=100,sigma=5)
ax.imshow(mask_pixel, cmap="Blues")
ax.imshow(mask11, alpha=0.5, cmap="Reds")

In [ ]:
plt.close("all")
%matplotlib inline

loader = SextantsNexusLoader(RAW_FOLDER)
series_data = {
    "workflow": "FTH_series",
    "reference_image_id": REFERENCE_IMAGE_ID,
    "calibration_file": REFERENCE_H5,
    "experimental_setup": experimental_setup,
    "positive_label": "+",
    "reference_label": "-",
    "hologram_labels": labels,
    "center": center,
    "mask_pixel": mask_pixel,
    "mask_beamstop_smooth_recipe": beamstop_recipe,
    "mask_pixel_fth_recipe": mask_pixel_fth_recipe,
    "focus_fth": {**focus_fth, "prop_dist_unit": "um"},
    "project_ewalds_sphere": project_ewalds_sphere,
    "ewald_method": ewald_method,
    "supportmask": calibration.get("supportmask"),
    "focus_cdi": calibration.get("focus_cdi", {}),
    "phase_retrieval_recipe": calibration.get("phase_retrieval_recipe"),
    "series": [],
}

for index, spec in enumerate(SERIES):


    
    # Copy recipe/geometry, then replace data belonging to the reference pair.
    entry = deepcopy({key: value for key, value in calibration.items()
                      if key not in ("holo", "recon", "recon_cdi", "phase_retrieval_errors",
                                     "phase_retrieval_png", "fth_png")})
    entry.update(index=index, point=spec.get("point", index), holo={},
                 workflow="FTH_series_point", calibration_file=REFERENCE_H5,
                 center=np.asarray(center), mask_pixel=mask_pixel,
                 focus_fth={**focus_fth, "prop_dist": prop_dist, "phase": phase,
                            "dx": dx, "dy": dy, "roi": roi, "prop_dist_unit": "um"})
    for label in labels:
        requested_ids = image_ids(spec[label])
        image_id = int(requested_ids[0])
        frame = load_average(loader, requested_ids)
        image = np.asarray(frame.image, dtype=float)

        #fig,ax=plt.subplots()
        #ax.set_title(requested_ids[0])
        #ax.imshow(image)
        #plt.show()

    
        dark_defaults = (calibration["holo"][label].get("dark_id")
                         if DARK_IDS is None else DARK_IDS[label])
        dark_id = spec.get(f"{label}_dark", dark_defaults)
        if dark_id is not None:
            dark = np.asarray(load_average(loader, image_ids(dark_id)).image, dtype=float)
            dark_recipe = calibration.get("dark_fit_recipe")
            if dark_recipe:
                detector_mask = None  # Fixed detector mask is disabled.
                image, _, _, _, _ = fit_dark_frame(
                    image, dark, slice(*dark_recipe["rows"]), slice(*dark_recipe["columns"]),
                    percentile=dark_recipe["percentile"], stride=dark_recipe["stride"],
                    mask=detector_mask,
                )
            else:
                image = image - dark
        centered = wf.center_image(image, center, cci)
        if project_ewalds_sphere:
            centered = PhR.inv_gnomonic(
                centered,
                center=np.asarray(centered.shape) / 2,
                experimental_setup=experimental_setup,
                method=ewald_method,
            )
        if centered.shape != mask_pixel.shape:
            raise ValueError(
                f"Series point {index}, {label}: shape {centered.shape} != calibration {mask_pixel.shape}"
            )
        entry["holo"][label] = {
            "id": requested_ids,
            "dark_id": dark_id,
            "source": str(frame.source),
            "exposure": frame.exposure,
            "raw_metadata": dict(frame.metadata),
            "image_c": np.asarray(centered),
        }

    pos_raw = np.asarray(entry["holo"]["+"]["image_c"], dtype=float)
    neg_raw = np.asarray(entry["holo"]["-"]["image_c"], dtype=float)
    band_recipe = dict(calibration.get("horizontal_band_recipe", {}))
    if band_recipe.pop("enabled", False):
        subtract_polynomial = band_recipe.pop("subtract_polynomial", False)
        corrected_pair = []
        for image in (pos_raw, neg_raw):
            fitted = fit_horizontal_band(image, mask=mask_pixel, **band_recipe)
            corrected_pair.append(image - fitted.band_image -
                                  (fitted.polynomial_image if subtract_polynomial else 0))
        pos_raw, neg_raw = corrected_pair
    if REUSE_CALIBRATION_SCALING:
        factor = float(calibration["factor"])
        offset = float(calibration["offset"])
    else:
        factor, offset = cci.dyn_factor(
            pos_raw * (1 - mask_pixel), neg_raw * (1 - mask_pixel),
            method="correlation", verbose=False, plot=False,
        )
        factor, offset = float(factor), float(offset)

    pos = pos_raw / factor
    neg = neg_raw
    holo = (pos - neg - offset) * mask_multiplier

    # wf.fth_reconstruct accepts micrometres and converts to metres once.
    recon = wf.reconstruct_fth_pair(
        pos * mask_multiplier, (neg + offset) * mask_multiplier,
        experimental_setup, fth, prop_dist=prop_dist, phase=phase, dx=dx, dy=dy,
        operation=focus_fth.get("operation", "-"),
    )
    entry["holo"]["+"]["image_c_norm"] = wf.normalize_image(pos)
    entry["holo"]["-"]["image_c_norm"] = wf.normalize_image(neg)
    entry["factor"] = factor
    entry["offset"] = offset
    entry["hologram"] = holo
    entry["recon"] = recon[roi_s]
    same_reference_inputs = all(
        image_ids(entry["holo"][label]["id"]) == image_ids(calibration["holo"][label]["id"])
        and np.array_equal(entry["holo"][label]["dark_id"], calibration["holo"][label].get("dark_id"))
        for label in labels
    )
    if (same_reference_inputs and REUSE_CALIBRATION_SCALING
            and np.array_equal(center, calibration["center"])
            and roi == list(calibration["focus_fth"]["roi"])
            and calibration.get("recon") is not None):
        expected = np.asarray(calibration["recon"])
        if expected.shape != entry["recon"].shape:
            raise ValueError("Reference reconstruction ROI does not match the series ROI")
        scale = max(float(np.linalg.norm(expected)), np.finfo(float).tiny)
        error = float(np.linalg.norm(entry["recon"] - expected) / scale)
        entry["reference_reconstruction_relative_error"] = error
        print(f"Reference reconstruction relative error: {error:.3g}")
        if not np.allclose(entry["recon"], expected, rtol=1e-9, atol=1e-9 * scale):
            raise ValueError(
                "The reference pair does not reproduce its saved FTH reconstruction. "
                "Check raw versus stitched inputs and regenerate the calibration with the current 01_FTH."
            )
    series_data["series"].append(entry)

    image_id = int(image_ids(entry["holo"]["+"]["id"])[0])
    png = join(BASEFOLDER, "processed", f"FTH_series_{index:04d}_ImId_{image_id:04d}_{USER}.png")
    fig, ax = plt.subplots(figsize=(5, 5))
    shown = np.real(entry["recon"])
    vmin, vmax = calibration.get(
        "fth_display_limits", wf.finite_percentile_limits(shown, (0.5, 99.5))
    )
    #vmin,vmax=-537573979.6310601,466776691.06362724
    #vmin,vmax=-9e8,9e8
    ax.imshow(shown, vmin=vmin, vmax=vmax, cmap="gray")
    if type(spec['+'])==list:
        ax.set_title(f"point {entry['point']}: +{spec['+'][0]} / -{spec['-'][0]}")
    else:
        ax.set_title(f"point {entry['point']}: +{spec['+']} / -{spec['-']}")
    ax.set_axis_off()
    fig.savefig(png, bbox_inches="tight", dpi=200)
    plt.close(fig)
    entry["fth_png"] = png

    # Checkpoint after each point so a long acquisition can be resumed/recovered.
    DATA_H5 = result_path(BASEFOLDER, image_id, USER, series=True)
    entry["data_file"] = DATA_H5
    wf.save_data_dict(entry, DATA_H5, overwrite=OVERWRITE)
    wf.save_data_dict(series_data, SERIES_H5, overwrite=OVERWRITE if index == 0 else True)
    print(DATA_H5)
    print(f"[{index + 1}/{len(SERIES)}] saved +{spec['+']} / -{spec['-']}")

print("Series HDF5:", SERIES_H5)

In [430]:
vmin*1e-8,vmax*1e-8

(-4.5944392149157975, 3.8139711332937476)

In [ ]:
fig,ax=plt.subplots()
ax.imshow(pos_raw, vmin=-1000,vmax=1000)

## Inspect the reconstructed loop

In [ ]:
plt.close("all")

stack = np.stack([entry["recon"] for entry in series_data["series"]])
print("Reconstruction stack:", stack.shape)

columns = min(4, len(series_data["series"]))
rows = int(np.ceil(len(series_data["series"]) / columns))
fig, axes = plt.subplots(columns, rows,
                         figsize=(3 * rows, 2. * columns),
                         squeeze=False)
for ax, entry in zip(axes.flat, series_data["series"]):
    shown = np.real(entry["recon"])
    vmin, vmax = calibration.get("fth_display_limits", wf.finite_percentile_limits(shown, (0.5, 99.5)))
    #vmin,vmax=-537573979.6310601,466776691.06362724
    ax.imshow(shown, vmin=vmin, vmax=vmax, cmap="gray")
    ax.set_title(f"+%s,-%s\n%dmT"%(entry['holo']["+"]["id"],entry['holo']["-"]["id"],entry["point"]))
    ax.set_axis_off()
for ax in axes.flat[len(series_data["series"]):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Create an animated GIF with one consistently scaled frame per reconstruction.
from PIL import Image

real_stack = np.real(stack)
finite = real_stack[np.isfinite(real_stack)]
if not finite.size:
    raise ValueError("Cannot create GIF: reconstructions contain no finite values")
vmin, vmax = calibration.get("fth_display_limits", np.percentile(finite, (0.5, 99.5)))
if vmax <= vmin:
    vmax = vmin + 1.0
gif_frames = []
for reconstruction in real_stack:
    normalized = np.clip((reconstruction - vmin) / (vmax - vmin), 0, 1)
    frame = np.nan_to_num(normalized, nan=0, posinf=1, neginf=0)
    gif_frames.append(Image.fromarray(np.uint8(np.rint(frame * 255))))

GIF_PATH = join(BASEFOLDER, "processed", f"FTH_series_ImId_{SERIES_IM_ID:04d}_{USER}.gif")
gif_frames[0].save(
    GIF_PATH,
    save_all=True,
    append_images=gif_frames[1:],
    duration=GIF_FRAME_DURATION_MS,
    loop=0,
)

series_data["fth_gif"] = GIF_PATH
series_data["gif_frame_duration_ms"] = GIF_FRAME_DURATION_MS
wf.save_data_dict(series_data, SERIES_H5, overwrite=True)
print("Saved GIF:", GIF_PATH)

In [343]:
SERIES_H5

'/home/experiences/sextants/com-sextants/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS-2609_SOLEIL_SEXTANTS/processed/Logs/data_recon_series_rb.hdf5'

In [344]:
# Acquisition ID summary
print("im_ids:", [{label: item["holo"][label].get("id") for label in item["holo"]} for item in series_data["series"]])
print("dark_ids:", [{label: item["holo"][label].get("dark_id") for label in item["holo"]} for item in series_data["series"]])

im_ids: [{'+': 1387, '-': np.int64(1388)}, {'+': 1391, '-': np.int64(1392)}, {'+': 1393, '-': np.int64(1394)}, {'+': 1395, '-': np.int64(1396)}, {'+': np.int64(1397), '-': np.int64(1398)}, {'+': np.int64(1399), '-': np.int64(1400)}, {'+': np.int64(1401), '-': np.int64(1402)}, {'+': np.int64(1403), '-': np.int64(1404)}, {'+': np.int64(1405), '-': np.int64(1406)}, {'+': np.int64(1407), '-': np.int64(1408)}, {'+': np.int64(1409), '-': np.int64(1410)}, {'+': np.int64(1411), '-': np.int64(1412)}, {'+': np.int64(1413), '-': np.int64(1414)}, {'+': np.int64(1415), '-': np.int64(1416)}, {'+': np.int64(1417), '-': np.int64(1418)}, {'+': np.int64(1419), '-': np.int64(1420)}, {'+': np.int64(1421), '-': np.int64(1422)}, {'+': np.int64(1423), '-': np.int64(1424)}, {'+': np.int64(1425), '-': np.int64(1426)}, {'+': np.int64(1427), '-': np.int64(1428)}, {'+': np.int64(1429), '-': np.int64(1430)}, {'+': np.int64(1431), '-': np.int64(1432)}, {'+': np.int64(1433), '-': np.int64(1434)}, {'+': np.int64(1435